# Lab 2 — Structured output (EP02)

In Lab 1 the teaching assistant could only answer with one label. This time we want a **4-field form** from each email:
intent (`intent`), urgency (`urgency`), course code (`course`) and a one-sentence summary (`summary`), so a program can use it right away

**What you'll learn** (the three steps from the theory video)
1. Ask for JSON in the prompt, and see how many answers pass `json.loads`
2. Write the schema as a class with **Pydantic** and validate every field
3. Send the schema to the API through `response_format`: the format is guaranteed, but **right shape ≠ right content**
4. Design a better schema: field descriptions, enums as short codes, our own validator, and asking again when it fails

The emails in this lab are **simulated data written for teaching**; every model answer comes from a real run.
(The video uses Thai emails; this English notebook uses English ones, so your outputs will differ in wording.)

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §1.12.11, §1.13.5, §19.3 — CC BY-SA 4.0

## 0. Set up before you start

Same as Lab 1: Python 3.10+ or Colab, and `OPENROUTER_API_KEY` in Colab Secrets or a `.env` file
(how to set up the key, and why we chose this model and provider, are in Lab 1)

In [ ]:
%pip install -q "openai>=1.50" python-dotenv "pydantic>=2.7"

In [ ]:
import os, json, re
from openai import OpenAI

def load_key():
    try:  # Google Colab Secrets
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
PROVIDER = {"order": ["parasail/bf16"], "allow_fallbacks": False}
EXTRA = {"reasoning": {"enabled": False}, "provider": PROVIDER}

def ask(messages, **kw):
    resp = client.chat.completions.create(model=MODEL, messages=messages, extra_body=EXTRA, **kw)
    return resp.choices[0].message.content, resp.usage.completion_tokens

In [ ]:
EMAILS = [
    "Dear lecturer, I have the flu. May I hand in homework 2 for AI301 three days late? I've attached a medical certificate.",
    "Which chapters does the midterm exam cover?",
    "The LMS crashed when I submitted my CS214 work, and I lost marks for being late. I don't think that's fair.",
    "Can we move the group report deadline to Monday? Our members have an exam in another course.",
    "Do we need our own API key for Lab 3 of AI301?",
    "I've submitted my homework twice but my score still shows zero. Could you check, please?",
]

## 1. Step one: ask for JSON in the prompt

Put the schema **before** the data (schema-first) and say "answer in JSON only", like the book's customer-email example

In [ ]:
PROMPT_JSON = """Read the student email and answer in JSON only
Schema:
{"intent": "question|extension|complaint", "urgency": "low|medium|high", "course": "string or null", "summary": "one sentence summary"}
Email: """
answers = [ask([{"role": "user", "content": PROMPT_JSON + email}], max_tokens=300)[0] for email in EMAILS]
print(answers[0])

In [ ]:
ok = 0
for text in answers:
    try:
        json.loads(text)
        ok += 1
    except json.JSONDecodeError as err:
        print("✗", err, "| starts with", repr(text[:8]))
print(f"json.loads passed {ok}/{len(answers)}")

In [ ]:
def strip_fences(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return m.group(0) if m else text

data = [json.loads(strip_fences(t)) for t in answers]
print(data[0])

## 2. Writing the schema as a class with Pydantic

`Literal[...]` = enum (only values from the list) · `str | None` = a string or null · `extra="forbid"` = no extra fields

In [ ]:
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

In [ ]:
class EmailInfo(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Literal["question", "extension", "complaint"]
    urgency: Literal["low", "medium", "high"]
    course: str | None
    summary: str

In [ ]:
for d in data:
    info = EmailInfo.model_validate(d)
    print(info.intent, info.urgency, repr(info.course))

In [ ]:
try:
    EmailInfo.model_validate({"intent": "extension", "urgency": "super urgent!!!"})
except ValidationError as err:
    print(err)

## 3. Step two: send the schema to the API

`response_format` of type `json_schema` + `strict: True` makes the provider enforce the format while generating (constrained decoding)
→ no backticks, no preamble, and one class works both ways

In [ ]:
def schema_of(model):
    return {"type": "json_schema",
            "json_schema": {"name": model.__name__, "strict": True, "schema": model.model_json_schema()}}

print(json.dumps(EmailInfo.model_json_schema()["properties"]["urgency"], ensure_ascii=False))

In [ ]:
results = []
for email in EMAILS:
    text, _ = ask([{"role": "user", "content": "Read the student email and extract the information\nEmail: " + email}],
                  max_tokens=300, response_format=schema_of(EmailInfo))
    info = EmailInfo.model_validate_json(text)
    results.append(info)
    print(info.intent, info.urgency, repr(info.course))

## 4. Designing a better schema

- `Field(description=...)` tells the model what each field means (it goes into the JSON schema too)
- enums as **short English codes** + descriptions: in the Thai version, Thai labels inside the schema made email six (score shows zero)
  come back as "extension" in all 4 test rounds, while English codes were right every time, 24/24
- `field_validator` checks what a schema can't express, e.g. a course code must be 2–3 capital letters followed by 3 digits

In [ ]:
class StudentEmail(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Literal["question", "extension", "complaint"] = Field(
        description="question = asks for information · extension = asks to hand in work late · complaint = reports a problem or is unhappy")
    urgency: Literal["low", "medium", "high"] = Field(description="high = a deadline or marks are at stake")
    course: str | None = Field(description="the course code written in the email, e.g. AI301; null if there is none")
    summary: str = Field(description="a one-sentence summary")

    @field_validator("course")
    @classmethod
    def must_be_code(cls, v):
        if v is not None and not re.fullmatch(r"[A-Z]{2,3}\d{3}", v):
            raise ValueError(f"not a course code: {v!r}")
        return v

In [ ]:
try:
    StudentEmail.model_validate({"intent": "question", "urgency": "low", "course": "not specified", "summary": "asks what the exam covers"})
except ValidationError as err:
    print(err.errors()[0]["msg"])

In [ ]:
SYSTEM = "You are a teaching assistant. Read the student email in <email> and fill in the schema using only information in the email"

def extract(email, tries=3):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"<email>\n{email}\n</email>"}]
    for attempt in range(1, tries + 1):
        text, _ = ask(msgs, max_tokens=300, response_format=schema_of(StudentEmail))
        try:
            return StudentEmail.model_validate_json(text)
        except ValidationError as err:
            print(f"  try {attempt} failed:", err.errors()[0]["msg"])
    return None

In [ ]:
LABEL = {"question": "Question", "extension": "Extension", "complaint": "Complaint"}
for email in EMAILS:
    info = extract(email)
    if info:
        print(LABEL[info.intent], info.urgency, repr(info.course), "|", info.summary)

## Summary

| Step | Result in this lab (Thai run in the video) |
|---|---|
| ask for JSON in the prompt | `json.loads` passed 0/6 — every answer wrapped in ```` ```json ```` |
| trim with a regex + Pydantic | 6/6 passed (but it's a patch) |
| `response_format` + a bare schema | right format every time, but course = "not specified" instead of null |
| field descriptions + English codes + validator + retry | every field right · course = a course code or None |

**Exercise:** add a field `deadline: str | None` (the date the student asks for) with a validator that only accepts `YYYY-MM-DD`,
then test it with emails that do and don't contain a date

▶ **Next: Lab 3 — Hallucination** — catch the model making things up, with repeated questions, token probabilities and grounding